# Wireless Networking and Mobile Computing

### 1. Transmitting GPS signals

#### 1.1 GPS orbit and frequencies

GPS satellites orbit the Earth at an altitude of about 20,200 km, completing two orbits per day. Their motion is carefully synchronized so that at least four satellites are visible from anywhere on Earth at all times. Each satellite continuously transmits radio signals toward Earth. Only the satellites transmit. Receivers on the ground are passive; they only listen to these signals. GPS uses several carrier frequencies on different radio bands:

| Band | Frequency (MHz) | Use | Notes |
|------|-----------------|------|-------|
| **L1** | 1575.42 | Civil | Used by all standard GPS receivers |
| **L2** | 1227.60 | Military (and dual-frequency civil) | Helps reduce atmospheric errors |
| **L5** | 1176.45 | Civil (aviation, safety-of-life) | Higher accuracy, modern systems |

These signals travel over 20,000 km, so they become very weak by the time they reach the ground, around –160 dBW, much weaker than background noise.  However, GPS receivers can still detect them thanks to their unique coded structure and correlation processing, which allows the weak signals to stand out.

#### 1.2 Navigation message content

Each GPS signal includes a navigation message that carries important information about the satellite and the system. This message is transmitted at a very low data rate (50 b/s) and repeats continuously. It contains the **satellite status and health**, **precise time information**, **orbital parameters (ephemeris)** describing the exact path of the satellite, and **approximate orbit data (almanac)** the coarse information for all satellites. This message allows the receiver to compute where each satellite was when its signal was transmitted, which is essential for accurate positioning.

The almanac tells a receiver roughly where and when each satellite will be visible, but not with high precision. Each satellite broadcasts the full almanac for the whole constellation. A receiver uses it for initial acquisition (finding satellites quickly), but for accurate positioning it needs the ephemeris data (precise orbit info for a single satellite).

| **Aspect**       | **Almanac** (approximate orbit data)                                       | **Ephemeris** (precise orbit data)             |
| ---------------- | -------------------------------------------------------------------------- | ---------------------------------------------- |
| **Scope**        | Info for *all satellites* in the constellation                             | Info for *one satellite* only                  |
| **Accuracy**     | Coarse (km-level) orbit + clock info                                       | Precise (meter-level) orbit + clock info       |
| **Size**         | Small dataset (~1 KB)                                                      | Larger dataset (~1–2 KB per satellite)         |
| **Validity**     | Days to weeks (stable over time)                                           | A few hours (must be refreshed often)          |
| **Transmission** | Sent by every satellite, for all satellites                                | Sent by each satellite, only for itself        |
| **Use**          | Helps receiver know *which satellites* to search for, and where in the sky | Needed for *accurate positioning* calculations |
| **Acquisition**  | Takes ~12.5 minutes to receive full set                                    | Takes ~30 seconds per satellite (good signal)  |

#### 1.3 PRN codes and signal modulation

Each GPS satellite uses a unique **Pseudo-Random Noise (PRN)** code to identify its signal. The main civil code is the **C/A (Coarse/Acquisition)** code, transmitted on the L1 frequency. It consists of 1,023 binary chips (+1 and –1) repeating every millisecond. Though it looks random, it is deterministic and known, so receivers can generate the same code locally.

Each satellite’s distinct PRN code allows all satellites to transmit on the same frequency using **Code Division Multiple Access (CDMA)**.
The transmitted GPS signal combines:

* the **carrier wave** (e.g., 1575.42 MHz for L1),
* the **PRN code**, which spreads the spectrum, and
* the **navigation message**, which carries timing and orbit data.

At the receiver, the signal is **correlated** with a locally generated PRN code. When alignment occurs, a sharp **correlation peak** reveals the signal delay, which gives the distance to that satellite.


#### 1.4 Demonstration: generating and correlating a PRN code

Below we simulate how a GPS receiver identifies a satellite signal.

**Setup**

In [ ]:
# --- Imports and Parameters ---
import numpy as np
import matplotlib.pyplot as plt

# Parameters
num_symbols = 6
samples_per_symbol = 1000
samples_per_chip = 100
fc = 100
fs = 10000
T_total = num_symbols * samples_per_symbol / fs
t = np.arange(0, T_total, 1/fs)
Rs = fs / samples_per_symbol
Rc = fs / samples_per_chip
np.random.seed(0)

# Plot style
axes_facecolor = (0.1, 0.1, 0.15)
line_color = {
    "message": "magenta",
    "prn": "deepskyblue",
    "spread": "red",
    "rx": "yellow",
    "baseband": "limegreen",
    "corr": "orange"
}

**Transmitter generates message and PRN Code**

In [ ]:
# --- Message and PRN code ---
# Generate
msg_symbols = np.array([1 if i % 2 == 0 else -1 for i in range(num_symbols)])
message = np.repeat(msg_symbols, samples_per_symbol)[:len(t)]

num_chips = int(T_total * Rc)
prn_chips = np.random.choice([1, -1], size=num_chips)
prn = np.repeat(prn_chips, samples_per_chip)[:len(t)]

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
plots = [
    (message, "Navigation Message", line_color["message"]),
    (prn, "PRN Code", line_color["prn"])
]

for ax, (data, title, color) in zip(axes, plots):
    ax.plot(t, data, color=color, lw=1.2)
    ax.set_title(title, pad=6)
    ax.set_facecolor(axes_facecolor)
    ax.grid(True, alpha=0.3)

plt.show()


**Transmitter spreads the message and modulates onto carrier**

In [ ]:
# --- Spread & Modulate ---
spread = message * prn
carrier = np.cos(2*np.pi*fc*t)
tx = spread * carrier

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
plots = [
    (spread, "Spread (message × PRN)", line_color["spread"]),
    (carrier, "Carrier cos(2πf_ct)", "white"),  # simple carrier
    (tx, "Transmitted RF (spread × carrier)", line_color["rx"]),
]

for ax, (sig, label, color) in zip(axes, plots):
    ax.plot(t, sig, color=color, lw=1.0)
    ax.set_title(label, pad=6)
    ax.set_facecolor(axes_facecolor)
    ax.grid(True, alpha=0.3)

plt.show()


**Propagation delays, adds noise and phase effects**

In [ ]:
# --- Simulate signal propagation ---
snr_db = 3.0
code_phase_offset = 12
carrier_phase = 0.4
freq_offset_hz = 0.0

prn_rx = np.roll(prn, code_phase_offset)
spread_rx = message * prn_rx
carrier_ch = np.cos(2*np.pi*(fc + freq_offset_hz)*t + carrier_phase)
tx_offset = spread_rx * carrier_ch

def add_awgn(x, snr_db):
    sig_power = np.mean(x**2)
    snr_lin = 10**(snr_db/10)
    noise_power = sig_power / snr_lin
    noise = np.sqrt(noise_power) * np.random.randn(*x.shape)
    return x + noise

rx = add_awgn(tx_offset, snr_db)

plt.figure(figsize=(10, 3))
plt.plot(t, rx, color=line_color["rx"], lw=1)
plt.title("Received Signal (delayed + noisy)", pad=6)
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor(axes_facecolor)
plt.show()

**Receiver removes carrier and despreads the signal**

In [ ]:
# ---  Demodulate & Despread ---
lo = np.cos(2*np.pi*fc*t)
baseband = rx * lo

plt.figure(figsize=(10, 3))
plt.plot(t, baseband, color=line_color["baseband"], lw=1)
plt.title("Baseband Signal after Carrier Wipeoff", pad=6)
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor(axes_facecolor)
plt.show()

**Receiver correlates with PRN to find delay and recover message**

In [ ]:
# --- Correlate ---
win_len = samples_per_symbol
metrics = np.zeros(samples_per_chip)

for tau in range(samples_per_chip):
    prn_hyp = np.roll(prn, tau)
    y = baseband[:win_len] * prn_hyp[:win_len]
    metrics[tau] = np.abs(np.sum(y))

est_code_phase = int(np.argmax(metrics))

# Convert code phase to time and distance
delta_t = est_code_phase / Rc                      # seconds
distance = 3e8 * delta_t                           # meters (speed of light)

print(f"Estimated code phase: {est_code_phase} samples")
print(f"Time delay: {delta_t:.3f} s")
print(f"Approx. pseudorange: {distance/1000:.3f} km\n")

# Despread and recover message
prn_aligned = np.roll(prn, est_code_phase)
despread = baseband * prn_aligned
despread_reshaped = despread[:num_symbols*samples_per_symbol].reshape(num_symbols, samples_per_symbol)
symbol_sums = np.sum(despread_reshaped, axis=1)
msg_hat = np.sign(symbol_sums)
msg_hat[msg_hat == 0] = 1
ber = np.mean(msg_hat != msg_symbols)

# Plot correlation result
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(samples_per_chip), metrics, color=line_color["corr"], lw=1.2)
ax.set_title(f"Correlation Search: estimated phase = {est_code_phase}", pad=6)
ax.set_xlabel("Code phase [samples]")
ax.set_ylabel("|Correlation|")
ax.set_facecolor(axes_facecolor)
ax.grid(True, alpha=0.3)
plt.show()


##### 1.4.2 Summary Plot

In [ ]:
# --- Combined visualization: full transmitter–receiver chain ---

fig, axes = plt.subplots(7, 1, figsize=(12, 16), sharex=False,
                         gridspec_kw={"hspace": 0.8})

def beautify(ax):
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_facecolor(axes_facecolor)

plots = [
    (message, "Message", line_color["message"]),
    (prn, "PRN", line_color["prn"]),
    (spread, "Spread (m×PRN)", line_color["spread"]),
    (rx, "Received (RF)", line_color["rx"]),
    (baseband, "Baseband (wipeoff)", line_color["baseband"]),
    (metrics, f"Correlation — est phase = {est_code_phase}", line_color["corr"]),
]

for i, (data, label, color) in enumerate(plots):
    axes[i].plot(data, color=color, lw=1)
    axes[i].set_title(label, loc="center", pad=6)
    beautify(axes[i])

# Final recovered message comparison
n = np.arange(num_symbols)
axes[6].step(n, msg_symbols, where='mid', lw=3, color="lime", label="Original")
axes[6].step(n, msg_hat, where='mid', lw=1.5, color="white", label="Recovered")
axes[6].set_title(f"Recovered Message (BER = {ber:.3f})", pad=6)
axes[6].set_xlabel("Symbol Index")
axes[6].set_ylabel("Bits")
axes[6].legend(fancybox=True, framealpha=1, facecolor=(0.6, 0.61, 0.8))
beautify(axes[6])

plt.show()


**Explanation:**

* Each **chip** in the PRN code lasts about **$1 \mu s$**, so the full 1,023-chip sequence transmitted at 1.023 Mcps repeats every $1 ms$.
* The **BPSK-modulated** signal is a sine wave whose phase flips according to the PRN sequence.
* The **correlation peak** appears when the locally generated code aligns with the received one, showing the **signal delay**, which corresponds to the travel time of the signal.

> Even with weak and noisy signals, correlation lets the receiver identify the satellite’s unique PRN pattern.
